# Chapter 7 — Project: Build & Train a ResNet

Companion notebook for **PyTorch From Ground Up, Volume 2, Chapter 7**.
Every code block printed in the chapter, in order, runnable top to bottom.

| Chapter section | Notebook section |
|---|---|
| The shortcut that does not fit | 1 |
| One block, written once | 2 |
| The architecture that fits CIFAR-10 | 3 |
| Reading the real thing | 4 |
| The experiment, and why it is set up this way | 5 |
| Degradation, measured | 6 |

Sections 1–5 run on a CPU in well under a minute. **Section 6 wants a GPU** —
about 1h40 on a Colab T4 for all five configurations at 40 epochs. `EPOCHS`
is a variable at the top of that cell.

Three things this notebook adds to what the chapter prints are listed in
`README.md`.

## 1 · The shortcut that does not fit

`F(x)` and `x` do not have the same shape at a stage boundary, so the
addition is illegal. The last line raises on purpose.

In [ ]:
import torch, torch.nn as nn
import torch.nn.functional as F

x = torch.randn(1, 16, 32, 32)
body = nn.Sequential(
    nn.Conv2d(16, 32, 3, stride=2, padding=1,
              bias=False),
    nn.BatchNorm2d(32), nn.ReLU(),
    nn.Conv2d(32, 32, 3, padding=1, bias=False),
    nn.BatchNorm2d(32))

print("x   ", tuple(x.shape))
print("F(x)", tuple(body(x).shape))
print(body(x) + x)

A 1×1 convolution at the same stride fixes the channels and the resolution together, for 576 parameters.

In [ ]:
short = nn.Sequential(
    nn.Conv2d(16, 32, 1, stride=2, bias=False),
    nn.BatchNorm2d(32))

print("short(x)", tuple(short(x).shape))
print("sum     ", tuple((body(x) + short(x)).shape))
print("params  ", sum(p.numel()
                      for p in short.parameters()))

## 2 · One block, written once

The `residual` flag is the experimental apparatus for section 6: setting it
to `False` deletes the addition and the projection and changes nothing
else.

In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, cin, cout, stride=1,
                 residual=True):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, stride,
                            padding=1, bias=False)
        self.b1 = nn.BatchNorm2d(cout)
        self.c2 = nn.Conv2d(cout, cout, 3,
                            padding=1, bias=False)
        self.b2 = nn.BatchNorm2d(cout)
        self.residual = residual
        self.short = None
        if residual and (stride != 1
                         or cin != cout):
            self.short = nn.Sequential(
                nn.Conv2d(cin, cout, 1, stride,
                          bias=False),
                nn.BatchNorm2d(cout))

    def forward(self, x):
        out = F.relu(self.b1(self.c1(x)))
        out = self.b2(self.c2(out))
        if self.residual:
            if self.short is not None:
                x = self.short(x)
            out = out + x
        return F.relu(out)

In [ ]:
cases = [(16, 16, 1, 32), (16, 32, 2, 32),
         (32, 64, 2, 16)]
for cin, cout, s, n in cases:
    blk = BasicBlock(cin, cout, s)
    out = blk(torch.randn(1, cin, n, n))
    kind = ("identity  " if blk.short is None
            else "projection")
    print(f"{cin:>3}->{cout:<3} s{s} {kind} "
          f"-> {tuple(out.shape)[1:]}")

## 3 · The architecture that fits CIFAR-10

He et al.'s section 4.2 family, depth 6*n*+2. `n=3` is ResNet-20, `n=9` is
ResNet-56.

In [ ]:
class CifarResNet(nn.Module):
    def __init__(self, n=3, residual=True,
                 n_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1,
                      bias=False),
            nn.BatchNorm2d(16), nn.ReLU())

        def stage(cin, cout, stride):
            b = [BasicBlock(cin, cout, stride,
                            residual)]
            b += [BasicBlock(cout, cout, 1, residual)
                  for _ in range(n - 1)]
            return nn.Sequential(*b)

        self.stage1 = stage(16, 16, 1)
        self.stage2 = stage(16, 32, 2)
        self.stage3 = stage(32, 64, 2)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(64, n_classes))

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        return self.head(x)

The shape trace. You could have written this out before running it.

In [ ]:
net = CifarResNet(n=3)
h = torch.randn(1, 3, 32, 32)
for name in ("stem", "stage1", "stage2", "stage3"):
    h = getattr(net, name)(h)
    print(f"{name:8s} {tuple(h.shape)}")
print(f"{'head':8s} {tuple(net.head(h).shape)}")

Where the parameters live. The classifier is 0.24% of the network — against VGG-16's 89.4%, which Chapter 4 measured.

In [ ]:
def count(m):
    return sum(p.numel() for p in m.parameters())

for label, m in [("stem", net.stem),
                 ("stage1", net.stage1),
                 ("stage2", net.stage2),
                 ("stage3", net.stage3),
                 ("head", net.head)]:
    print(f"{label:8s} {count(m):>8,d}")

## 4 · Reading the real thing

`conv1` is the 9,408-parameter stem counted in Chapter 3. `downsample` is the
projection shortcut you wrote in section 1; torchvision uses a different name
for it.

In [ ]:
from torchvision.models import resnet18

r18 = resnet18(weights=None)
print(r18.conv1)
print(r18.layer1[0].conv1)
print(r18.layer2[0].downsample)

### Not printed in the chapter: the bottleneck block

The chapter's Try It box asks you to compare `resnet50.layer1[0]` with
`resnet18.layer1[0]`. Here it is, with the parameter arithmetic Chapter 6
costed.

In [ ]:
from torchvision.models import resnet50

r50 = resnet50(weights=None)
print(r50.layer1[0])
print()

# Chapter 6's arithmetic, 256 -> 256:
full = 2 * (256 * 256 * 9)
neck = 256 * 64 + 64 * 64 * 9 + 64 * 256
print(f"two full 3x3 blocks : {full:,}")
print(f"1x1 - 3x3 - 1x1     : {neck:,}")
print(f"ratio               : {full / neck:.1f}x")

## 5 · The experiment, and why it is set up this way

One recipe for all five networks, so the only variable in the results table
is the architecture. SGD rather than Adam: Adam adapts a learning rate per
parameter, which is exactly what would mask a badly-conditioned deep plain
network — the failure this chapter is trying to see.

In [ ]:
EPOCHS = 40
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)

In [ ]:
model = CifarResNet(n=3).to(dev)
decay = [p for p in model.parameters()
         if p.ndim > 1]
flat = [p for p in model.parameters()
        if p.ndim <= 1]
opt = torch.optim.SGD(
    [{"params": decay, "weight_decay": 5e-4},
     {"params": flat, "weight_decay": 0.0}],
    lr=0.1, momentum=0.9)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt, T_max=EPOCHS)

## 6 · Degradation, measured

**This cell needs a GPU.** Five configurations at 40 epochs is roughly 1h40
on a Colab T4 — BetterCNN 14 min, plain-20 and ResNet-20 about 17 each, the
two 56-layer runs about 26 each.

Lower `EPOCHS` for a quick look, but the result genuinely needs the full run:
plain-56 is still climbing at epoch 40 and a short run understates how far
behind it is.

The version that checkpoints after every epoch and survives a Colab
disconnect is the one the chapter's Watch Out box describes: save model,
optimiser, scheduler and curves every epoch, to a temporary name that is then
renamed over the real one. This one is the
straightforward reading of the same experiment.

### Not printed in the chapter: `BetterCNN`

Chapter 6's model, reproduced here so the comparison row can actually be
run.

In [ ]:
from torchvision import datasets, transforms


class BetterCNN(nn.Module):
    """Chapter 6's model, unchanged."""

    def __init__(self, n_classes=10):
        super().__init__()

        def stage(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, padding=1, bias=False),
                nn.BatchNorm2d(cout), nn.ReLU(), nn.MaxPool2d(2))

        self.body = nn.Sequential(stage(3, 32), stage(32, 64),
                                  stage(64, 128))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Linear(128, n_classes))

    def forward(self, x):
        return self.head(self.body(x))

In [ ]:
NORM = transforms.Normalize([0.5] * 3, [0.5] * 3)
aug_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), NORM])
test_tf = transforms.Compose([transforms.ToTensor(), NORM])

train_set = datasets.CIFAR10("data", train=True, download=True,
                             transform=aug_tf)
train_eval = datasets.CIFAR10("data", train=True, download=True,
                              transform=test_tf)
test_set = datasets.CIFAR10("data", train=False, download=True,
                            transform=test_tf)

train_loader = torch.utils.data.DataLoader(
    train_set, batch_size=128, shuffle=True, num_workers=2,
    pin_memory=(dev == "cuda"))
eval_loaders = {
    "train": torch.utils.data.DataLoader(train_eval, batch_size=500),
    "test": torch.utils.data.DataLoader(test_set, batch_size=500),
}
print(len(train_set), "train /", len(test_set), "test")

In [ ]:
import time


@torch.no_grad()
def accuracy(model, loader):
    model.eval()
    right = total = 0
    for xb, yb in loader:
        xb, yb = xb.to(dev), yb.to(dev)
        right += (model(xb).argmax(1) == yb).sum().item()
        total += len(yb)
    return right / total


def build(kind, n):
    torch.manual_seed(0)
    if kind == "bcnn":
        model = BetterCNN()
    else:
        model = CifarResNet(n=n, residual=(kind == "res"))
    model = model.to(dev)
    decay = [p for p in model.parameters() if p.ndim > 1]
    flat = [p for p in model.parameters() if p.ndim <= 1]
    opt = torch.optim.SGD(
        [{"params": decay, "weight_decay": 5e-4},
         {"params": flat, "weight_decay": 0.0}],
        lr=0.1, momentum=0.9)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    return model, opt, sched


def run(label, kind, n):
    model, opt, sched = build(kind, n)
    t0 = time.time()
    for ep in range(EPOCHS):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(dev), yb.to(dev)
            loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
        sched.step()
        if (ep + 1) % 10 == 0 or ep == EPOCHS - 1:
            print(f"  {label:10s} epoch {ep+1:2d}  "
                  f"test {accuracy(model, eval_loaders['test']):.4f}",
                  flush=True)
    tr = accuracy(model, eval_loaders["train"])
    te = accuracy(model, eval_loaders["test"])
    n_p = sum(p.numel() for p in model.parameters())
    print(f"{label:10s} {n_p:>9,}  train err {(1-tr)*100:5.1f}%  "
          f"test {te*100:5.1f}%  ({time.time()-t0:.0f}s)")
    return dict(label=label, params=n_p, train=tr, test=te)


results = []
for label, kind, n in [("BetterCNN", "bcnn", 0),
                       ("plain-20", "plain", 3),
                       ("ResNet-20", "res", 3),
                       ("plain-56", "plain", 9),
                       ("ResNet-56", "res", 9)]:
    results.append(run(label, kind, n))

In [ ]:
print(f"{'model':<12}{'params':>10}{'train err':>11}{'test':>8}")
for r in results:
    print(f"{r['label']:<12}{r['params']:>10,}"
          f"{(1-r['train'])*100:>10.1f}%{r['test']*100:>7.1f}%")

by = {r["label"]: r for r in results}
d = (1 - by["plain-56"]["train"]) - (1 - by["plain-20"]["train"])
print(f"\nplain-56 minus plain-20 on TRAINING error: {d*100:+.1f} points")
print("Positive is the degradation problem. That is the whole experiment.")

## Reproducing the book's numbers

The book's run: torch 2.11.0+cu128, Colab T4, `manual_seed(0)`, SGD lr 0.1,
momentum 0.9, cosine to zero, weight decay 5e-4 on parameters with
`ndim > 1`, `RandomCrop(32, padding=4)` + horizontal flip, batch 128,
40 epochs.

| Model | Params | Train err. | Train acc. | Test acc. |
|---|---|---|---|---|
| BetterCNN | 94,762 | 11.9% | 88.1% | **84.3%** |
| plain-20 | 269,722 | 4.9% | 95.1% | **88.7%** |
| plain-56 | 853,018 | **16.0%** | 84.0% | **79.9%** |
| ResNet-20 | 272,474 | 2.6% | 97.4% | **90.4%** |
| ResNet-56 | 855,770 | **1.0%** | 99.0% | **91.7%** |

Expect a few tenths either way — cuDNN does not pick the same kernels on
every machine and the seed does not reach that far. The *shape* of the table
should hold, and the two numbers that matter are in bold in the training
error column: **plain-56 fits its own training data 11.1 points worse than
plain-20, while ResNet-56 fits it 1.6 points better than ResNet-20.**

Note that BetterCNN reaches 84.3% here against the 79.6% Chapter 6 reported.
Nothing about the model changed; the optimiser did, from Adam at 1e-3 to SGD
at 0.1 with a cosine schedule. That is why it is retrained in this chapter
rather than quoted — and it is a fair warning about comparing numbers across
recipes.